# Whisper Medium LoRA Levantine Custom Run: Palestine + Jordan + North Levantine

This notebook mirrors the practical structure of the original Whisper Medium mini run, but swaps in the exact Levantine datasets requested for training and held-out evaluation.

Custom dataset plan:
- train on the requested Casablanca Palestine/Jordan validation parquet files
- train on the requested Omnilingual APC North Levantine Arrow files (`data-00000`, `data-00001`)
- split the requested Casablanca test parquet files 50/50 into validation and test
- split Omnilingual `data-00002-of-00003.arrow` 50/50 into validation and test
- train for up to 50 epochs with early stopping patience `3` on rising validation loss


## 1. Config

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path("/home/MohammadNabulsi/whisper")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    RUN_DIR,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = True
TRAIN_HOURS_CAP = 1.0
EVAL_SAMPLE_CAP = 1
RUN_BASELINE_BEFORE_TRAIN = True
RUN_POST_TRAIN_EVAL = True

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    train_hours_cap=TRAIN_HOURS_CAP,
    eval_sample_cap=EVAL_SAMPLE_CAP,
    num_train_epochs=50,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
)
ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print("Notebook path:", SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print("Log path:", log_path)


## 2. Build Filtered Mini Manifests

In [ ]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import prepare_manifests

manifest_state = prepare_manifests(config)
selection_summary = manifest_state["selection_summary"]
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


## 3. Baseline Whisper Medium Predictions

In [ ]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import load_whisper_bundle, run_predictions

baseline_bundle = load_whisper_bundle(config)
baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state["test_rows"],
        config,
        baseline_bundle,
        name="baseline_test_predictions",
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


## 4. Train LoRA Adapters

In [ ]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import train_model

training_summary = train_model(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


## 5. Tuned Validation and Test Predictions

In [ ]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import load_whisper_bundle, run_predictions

tuned_bundle = load_whisper_bundle(config, adapter_path=training_summary["best_checkpoint"])
val_prediction_metrics = run_predictions(
    manifest_state["val_rows"],
    config,
    tuned_bundle,
    name="tuned_val_predictions",
) if config.run_post_train_eval else None
test_prediction_metrics = run_predictions(
    manifest_state["test_rows"],
    config,
    tuned_bundle,
    name="tuned_test_predictions",
) if config.run_post_train_eval else None
print("Validation metrics:")
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print("Test metrics:")
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))


## 6. Summary Report

In [ ]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))


## 7. Integrity Report

In [ ]:
from Runs.whisper_medium_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report

integrity_report = write_integrity_report(
    config,
    selection_summary,
    baseline_test_metrics,
    val_prediction_metrics,
    test_prediction_metrics,
    training_summary,
)
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))
if integrity_report.get("prediction_health", {}).get("test_predictions", {}).get("object_dump_predictions", 0):
    raise RuntimeError("Decode guard failed: found object-dump predictions in test output.")
if integrity_report.get("prediction_health", {}).get("test_predictions", {}).get("empty_predictions", 0):
    raise RuntimeError("Integrity check failed: found empty predictions in test output.")
print("WHISPER MEDIUM MINI RUN INTEGRITY CHECK PASSED")
